In [ ]:
#| default_exp renderers.nanobanana

# renderers.nanobanana

> Renderer for NanaBanana (Google Imagen via `google-genai`).
>
> Supports reference image input for character consistency.
> API key: `GOOGLE_API_KEY` environment variable.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
import os
from pathlib import Path

from manhualizer.config import OutputConfig, RendererConfig
from manhualizer.models import Panel, RenderResult
from manhualizer.render import BaseRenderer, ModelSpec

In [ ]:
#| export
class NanaBananaRenderer(BaseRenderer):
    """Image generation via Google Imagen (NanaBanana).

    Capabilities: reference image input, multi-image input.
    No LoRA, no negative prompt support.

    Requires: GOOGLE_API_KEY environment variable.
    """

    def __init__(self, model_spec: ModelSpec, config: RendererConfig):
        super().__init__(model_spec, config)
        self._model_cfg = config.nanobanana

    def _client(self):
        """Lazy-initialise google-genai client."""
        import google.generativeai as genai  # type: ignore
        api_key = os.environ.get("GOOGLE_API_KEY")
        if not api_key:
            raise EnvironmentError("GOOGLE_API_KEY is not set")
        genai.configure(api_key=api_key)
        return genai

    async def render_async(
        self,
        panel: Panel,
        output_dir: Path,
        output_cfg: OutputConfig,
        reference_images: dict[str, Path] | None = None,
    ) -> RenderResult:
        import asyncio
        return await asyncio.get_event_loop().run_in_executor(
            None, self._render_sync, panel, output_dir, output_cfg, reference_images
        )

    def _render_sync(
        self,
        panel: Panel,
        output_dir: Path,
        output_cfg: OutputConfig,
        reference_images: dict[str, Path] | None,
    ) -> RenderResult:
        from google.generativeai import ImageGenerationModel  # type: ignore

        genai = self._client()
        model = genai.ImageGenerationModel(self._model_cfg.model)

        prompt = panel.visual_prompt
        w, h = output_cfg.resolved_dimensions()

        # Build reference image list for characters present in this panel
        ref_image_parts = []
        if reference_images:
            for char_name in panel.characters_present:
                ref_path = reference_images.get(char_name)
                if ref_path and ref_path.exists():
                    ref_image_parts.append({"mime_type": "image/png", "data": ref_path.read_bytes()})

        generate_kwargs: dict = {
            "prompt": prompt,
            "number_of_images": 1,
        }
        if ref_image_parts:
            generate_kwargs["reference_images"] = ref_image_parts

        result = model.generate_images(**generate_kwargs)
        image = result.images[0]

        out_path = output_dir / f"panel_{panel.panel_number:04d}.{output_cfg.format}"
        image.save(str(out_path))

        return RenderResult(
            panel_number=panel.panel_number,
            image_path=out_path,
            backend_used=self.model_spec.name,
            prompt_used=prompt,
            metadata={"model": self._model_cfg.model, "ref_images_used": len(ref_image_parts)},
        )

In [ ]:
# Construction test (no API call)
from manhualizer.render import MODELS
from manhualizer.config import RendererConfig
from manhualizer.renderers.nanobanana import NanaBananaRenderer

renderer = NanaBananaRenderer(MODELS["nanobanana"], RendererConfig())
assert renderer.model_spec.capabilities.reference_images
assert renderer.model_spec.capabilities.multi_image_input
assert not renderer.model_spec.capabilities.lora
print("NanaBananaRenderer OK")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()